# Feature engineering

Primero haremos modelos para cada familia de productos y ver que tal funciona

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [10]:
TRAIN_CSV_PATH = '../data/raw/store-sales-time-series-forecasting/train.csv'

In [28]:
train = pd.read_csv(TRAIN_CSV_PATH, parse_dates=['date'])
train['date'] = pd.to_datetime(train['date'])

In [29]:
train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [30]:
grouped_train = train.groupby(['family', 'date']).agg({'sales': 'sum', 'onpromotion': 'sum'}).reset_index()

grouped_train.head()

,family,date,sales,onpromotion
0,AUTOMOTIVE,2013-01-01,0.0,0
1,AUTOMOTIVE,2013-01-02,255.0,0
2,AUTOMOTIVE,2013-01-03,161.0,0
3,AUTOMOTIVE,2013-01-04,169.0,0
4,AUTOMOTIVE,2013-01-05,342.0,0


In [31]:
families = grouped_train['family'].value_counts().index

In [32]:
dataframes = {}

for family in families:
    family_df = grouped_train[grouped_train['family'] == family].copy()
    family_df.set_index('date', inplace=True)
    family_df = family_df.asfreq('D')
    dataframes[family] = family_df

dataframes['AUTOMOTIVE'].head()

,family,sales,onpromotion
date,,,
2013-01-01,AUTOMOTIVE,0.0,0.0
2013-01-02,AUTOMOTIVE,255.0,0.0
2013-01-03,AUTOMOTIVE,161.0,0.0
2013-01-04,AUTOMOTIVE,169.0,0.0
2013-01-05,AUTOMOTIVE,342.0,0.0


In [39]:
#Ver si hay valores nulos en los dataframes
for family, df in dataframes.items():
    if df.isnull().values.any():
        print(f"Missing values found in {family} dataframe.")
    else:
        print(f"No missing values in {family} dataframe.")

No missing values in AUTOMOTIVE dataframe.
No missing values in BABY CARE dataframe.
No missing values in BEAUTY dataframe.
No missing values in BEVERAGES dataframe.
No missing values in BOOKS dataframe.
No missing values in BREAD/BAKERY dataframe.
No missing values in CELEBRATION dataframe.
No missing values in CLEANING dataframe.
No missing values in DAIRY dataframe.
No missing values in DELI dataframe.
No missing values in EGGS dataframe.
No missing values in FROZEN FOODS dataframe.
No missing values in GROCERY I dataframe.
No missing values in GROCERY II dataframe.
No missing values in HARDWARE dataframe.
No missing values in HOME AND KITCHEN I dataframe.
No missing values in HOME AND KITCHEN II dataframe.
No missing values in HOME APPLIANCES dataframe.
No missing values in HOME CARE dataframe.
No missing values in LADIESWEAR dataframe.
No missing values in LAWN AND GARDEN dataframe.
No missing values in LINGERIE dataframe.
No missing values in LIQUOR,WINE,BEER dataframe.
No missin

In [40]:
#Rellenamos family con cada familia, sales con la media y onpromotion con 0
for family, df in dataframes.items():
    df['family'] = family
    df['sales'] = df['sales'].fillna(df['sales'].mean())
    df['onpromotion'] = df['onpromotion'].fillna(0)

dataframes['AUTOMOTIVE'].head()

,family,sales,onpromotion
date,,,
2013-01-01,AUTOMOTIVE,0.0,0.0
2013-01-02,AUTOMOTIVE,255.0,0.0
2013-01-03,AUTOMOTIVE,161.0,0.0
2013-01-04,AUTOMOTIVE,169.0,0.0
2013-01-05,AUTOMOTIVE,342.0,0.0


In [43]:
# Pasamos con stats models una funcion determinista para la tendencia solo de grado 1 
from statsmodels.tsa.deterministic import DeterministicProcess

for family, df in dataframes.items():
    dp = DeterministicProcess(index=df.index, order=1, drop=True)
    df['trend'] = dp.in_sample()

dataframes['AUTOMOTIVE'].head()

,family,sales,onpromotion,trend
date,,,,
2013-01-01,AUTOMOTIVE,0.0,0.0,1.0
2013-01-02,AUTOMOTIVE,255.0,0.0,2.0
2013-01-03,AUTOMOTIVE,161.0,0.0,3.0
2013-01-04,AUTOMOTIVE,169.0,0.0,4.0
2013-01-05,AUTOMOTIVE,342.0,0.0,5.0


In [44]:
#Hacemos el target de cada una de las familias con los sales y los quitamos de los features
dataframes_targets = {}

for family, df in dataframes.items():
    dataframes_targets[family] = df['sales']
    df.drop(columns=['sales'], inplace=True)

dataframes_targets['AUTOMOTIVE'].head()

date
2013-01-01      0.0
2013-01-02    255.0
2013-01-03    161.0
2013-01-04    169.0
2013-01-05    342.0
Freq: D, Name: sales, dtype: float64